In [5]:
import torch
import os
import torch.nn.functional as F
from collections import defaultdict
import json

In [6]:
def token_nonsimilarity_score_abs(
    sim: torch.Tensor,
    p: float = 3.0,
    type_fn: str = 'p'
) -> torch.Tensor:

    assert sim.ndim == 3, f"Expected 3D tensor [steps, layers, tokens], got shape {tuple(sim.shape)}"

    diff = torch.abs(1.0 - sim)
    if type_fn == 'p':
        score = diff.pow(p).mean(dim=(0, 1)).pow(1.0 / p)
    elif type_fn == 'log':
        score = torch.log1p(diff).mean(dim=(0, 1))
    # end

    return score
# end

In [7]:
def load_sim_matrix_and_transform_to_most_diff_list(folder_kv_base, filename, num_block, len_prompt, size_block, type_fn='p'):
    path_kv_file = os.path.join(folder_kv_base, filename)
    matrix_sim_step_layer_token = torch.load(path_kv_file)
    matrix_sim_step_layer_token = F.pad(matrix_sim_step_layer_token, (0,0,0,0,1,0), value=1.0)

    list_idx_sim_sorted = []

    for id_block in range(num_block):
        position_start = len_prompt + size_block * id_block

        matrix_sim_step_layer_token_cached = matrix_sim_step_layer_token[id_block*size_block:(id_block+1)*size_block, :, :position_start]  #(steps_block, len_cached)

        score_diff_token = token_nonsimilarity_score_abs(matrix_sim_step_layer_token_cached, type_fn=type_fn)
        idx_diff_token_decending = torch.argsort(score_diff_token, descending=True)
        list_idx_sim_sorted.append({'filename': filename, 'block': id_block, 'idx': idx_diff_token_decending.tolist(), 'value': score_diff_token[idx_diff_token_decending].tolist()})
    # end

    return list_idx_sim_sorted
# end

In [8]:
folder_kv_base = 'sims_kv'
type_fn = 'p'
filename_report = f'all_in_one_sim_report_abs_{type_fn}.json'

len_prompt = 128
num_block = 4
len_target = 256
size_block = int(len_target / num_block)

dict_filename_to_list_idx_sorted = defaultdict(list)

for filename in os.listdir(folder_kv_base):
    if filename[0] == '.':
        continue
    # end

    # matrix_sim_step_layer_token, num_block, len_prompt, size_block, path_kv_base, filename
    list_diff_sorted = load_sim_matrix_and_transform_to_most_diff_list(folder_kv_base, filename, num_block, len_prompt, size_block, type_fn)
    dict_filename_to_list_idx_sorted[filename] = list_diff_sorted
# end

with open(filename_report, 'w+') as file:
    file.write(json.dumps(dict_filename_to_list_idx_sorted))
# end

